## SHAP Explainability for the VGG16 Chest X-ray Classifier

Loads the checkpoint saved by vgg16_chest_xray.py and uses SHAP's GradientExplainer to visualize which pixels pushed the model toward NORMAL vs PNEUMONIA for a handful of test images.

This is a standalone script. It does NOT import vgg16_chest_xray.py, because that file runs the data split and a full training run as soon as it's executed. Only the few settings needed to rebuild the model and preprocess images the same way are duplicated here.

Requires: pip install shap Run this after models/vgg16_best.pth exists.


## 1. IMPORTS


In [1]:
from pathlib import Path

from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import shap
import torch
from torch import nn
from torchvision import datasets, models, transforms


/home/mega-pc/pneumo-xai/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. CONFIGURATION

Mirrors vgg16_chest_xray.py's settings. Kept in sync manually, not imported - see the note above.


In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "vgg16_best.pth"
EXPLANATIONS_DIR = PROJECT_ROOT / "explanations"
LOG_PATH = PROJECT_ROOT / "logs" / "shap_runs.log"

IMAGE_SIZE = 224
NUM_CLASSES = 2
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

N_BACKGROUND = 50     # background images SHAP uses as its reference distribution
N_CORRECT = 4          # correctly predicted test images to visualize
N_MISCLASSIFIED = 4    # misclassified test images to visualize (if available)
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## 3. REBUILD THE MODEL AND LOAD THE TRAINED WEIGHTS


In [3]:
def build_model():
    model = models.vgg16(weights=None)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)
    return model


def disable_inplace_relu(model):
    # VGG16's ReLU layers default to inplace=True, which breaks
    # gradient-based explainers (SHAP, Captum) because the
    # original activations get overwritten before the backward
    # pass can use them. This switches every ReLU to out-of-place.
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False


checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = build_model().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
disable_inplace_relu(model)
model.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Validation accuracy at save time: {checkpoint['val_accuracy']:.4f}")


Loaded checkpoint from epoch 10
Validation accuracy at save time: 0.9555


## 4. DATA FOR SHAP

Background set: a small random sample of TRAINING images. SHAP uses this as a reference ("what does a typical input look like") to measure each pixel's contribution against.

Explain set: a few TEST images to actually visualize.

Same preprocessing as training's eval_transform: resize only, no augmentation, no normalization.


In [4]:
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=eval_transform)

rng = np.random.default_rng(SEED)

background_indices = rng.choice(len(train_dataset), size=N_BACKGROUND, replace=False)
background = torch.stack(
    [train_dataset[i][0] for i in background_indices]
).to(DEVICE)

# Predict the whole test set once, so misclassified examples can be
# sampled deliberately instead of hoping a random draw includes one.
all_test_images = torch.stack([image for image, _ in test_dataset])
all_test_labels = torch.tensor([label for _, label in test_dataset])

with torch.no_grad():
    all_predictions = []
    for start in range(0, len(all_test_images), 32):
        batch = all_test_images[start:start + 32].to(DEVICE)
        all_predictions.append(model(batch).argmax(dim=1).cpu())
    all_predictions = torch.cat(all_predictions)

correct_mask = all_predictions == all_test_labels
correct_indices = torch.nonzero(correct_mask).flatten().tolist()
misclassified_indices = torch.nonzero(~correct_mask).flatten().tolist()

# Overall accuracy can hide a model that just leans on the
# majority class. Breaking it down per true class (recall) and
# per predicted class (precision) shows whether errors are
# spread evenly or concentrated on one class.
confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
for true_class, predicted_class in zip(all_test_labels.tolist(), all_predictions.tolist()):
    confusion[true_class, predicted_class] += 1

print("\nConfusion matrix (rows=true, columns=predicted):")
print("            " + "  ".join(f"{name:>10}" for name in CLASS_NAMES))
for true_class, name in enumerate(CLASS_NAMES):
    row = "  ".join(f"{confusion[true_class, pred_class].item():>10}" for pred_class in range(NUM_CLASSES))
    print(f"{name:>10}  {row}")

for true_class, name in enumerate(CLASS_NAMES):
    support = confusion[true_class].sum().item()
    recall = confusion[true_class, true_class].item() / support
    print(f"Recall on true {name}: {recall:.2%} ({confusion[true_class, true_class].item()}/{support})")

n_correct = min(N_CORRECT, len(correct_indices))
n_misclassified = min(N_MISCLASSIFIED, len(misclassified_indices))
test_misclassified_count = len(misclassified_indices)
test_total_count = len(all_test_labels)
test_misclassified_rate = test_misclassified_count / test_total_count
print(
    f"Test set: {test_misclassified_count} misclassified out of "
    f"{test_total_count} ({test_misclassified_rate:.2%})"
)

explain_indices = list(rng.choice(correct_indices, size=n_correct, replace=False)) + \
    list(rng.choice(misclassified_indices, size=n_misclassified, replace=False))
explain_images = torch.stack(
    [test_dataset[i][0] for i in explain_indices]
).to(DEVICE)
explain_labels = [test_dataset[i][1] for i in explain_indices]



Confusion matrix (rows=true, columns=predicted):
                NORMAL   PNEUMONIA
    NORMAL         212          26
 PNEUMONIA           6         636
Recall on true NORMAL: 89.08% (212/238)
Recall on true PNEUMONIA: 99.07% (636/642)
Test set: 32 misclassified out of 880 (3.64%)


## 5. RUN SHAP

GradientExplainer returns one array of per-pixel contributions per output class, each shaped (N_EXPLAIN, 3, 224, 224).


In [5]:
explainer = shap.GradientExplainer(model, background, local_smoothing=0.15)
shap_values_raw = explainer.shap_values(explain_images)

# Older shap versions return a list with one array per class,
# each shaped (N, C, H, W). Newer versions return a single
# array shaped (N, C, H, W, num_classes) instead. Normalize to
# the list-of-per-class-arrays form either way.
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw
else:
    shap_values = [
        shap_values_raw[..., class_index]
        for class_index in range(shap_values_raw.shape[-1])
    ]


## 6. VISUALIZE

Shows each explained image next to two one-color panels: - Column 2 (red): ONLY the pixels that pushed toward NORMAL. Negative/away-from-NORMAL pixels are hidden. - Column 3 (blue): ONLY the pixels that pushed toward PNEUMONIA. Negative/away-from-PNEUMONIA pixels are hidden. Both columns share the same intensity scale, so a direct comparison of red vs blue tells you which class had the stronger supporting evidence at a given spot.


In [6]:
with torch.no_grad():
    predicted_classes = model(explain_images).argmax(dim=1).cpu().numpy()

image_result_lines = []

for i in range(len(explain_indices)):
    true_label = CLASS_NAMES[explain_labels[i]]
    pred_label = CLASS_NAMES[predicted_classes[i]]
    print(f"Image {i}: true={true_label}, predicted={pred_label}")
    image_result_lines.append(f"  Image {i}: true={true_label}, predicted={pred_label}")

# Channel-last images in [0, 1], one per explained example.
image_numpy = explain_images.detach().cpu().numpy().transpose(0, 2, 3, 1)

# Collapse each class's per-channel attribution into one signed
# value per pixel, then keep only the positive part - i.e. only
# the evidence that supports that column's own class.
normal_evidence = [np.clip(shap_values[0][row].sum(axis=0), 0, None) for row in range(len(explain_indices))]
pneumonia_evidence = [np.clip(shap_values[1][row].sum(axis=0), 0, None) for row in range(len(explain_indices))]

# Shared scale across both columns, so color strength is directly
# comparable between "evidence for NORMAL" and "evidence for
# PNEUMONIA". Using a high percentile instead of the true max:
# gradient-based SHAP maps usually have a few extreme outlier
# pixels, and scaling to the true max would wash every normal
# pixel out to near-white. Percentile clipping keeps the scale
# driven by the typical strong pixels instead of one spike.
all_evidence = np.concatenate(
    [m.flatten() for m in normal_evidence + pneumonia_evidence]
)
shared_vmax = np.percentile(all_evidence, 99.5)

num_rows = len(explain_indices)
fig, axes = plt.subplots(num_rows, 3, figsize=(9, 3 * num_rows))
if num_rows == 1:
    axes = axes[np.newaxis, :]

for row in range(num_rows):
    true_label = CLASS_NAMES[explain_labels[row]]
    pred_label = CLASS_NAMES[predicted_classes[row]]

    axes[row, 0].imshow(image_numpy[row])
    axes[row, 0].set_title(f"true={true_label}\npred={pred_label}", fontsize=9)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(image_numpy[row])
    axes[row, 1].imshow(
        normal_evidence[row], cmap="Reds", alpha=0.8, vmin=0, vmax=shared_vmax
    )
    axes[row, 1].set_title("Evidence for NORMAL", fontsize=9)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(image_numpy[row])
    axes[row, 2].imshow(
        pneumonia_evidence[row], cmap="Blues", alpha=0.8, vmin=0, vmax=shared_vmax
    )
    axes[row, 2].set_title("Evidence for PNEUMONIA", fontsize=9)
    axes[row, 2].axis("off")

plt.tight_layout()

EXPLANATIONS_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = (
    EXPLANATIONS_DIR
    / f"shap_epoch{checkpoint['epoch']}_{timestamp}.png"
)
plt.savefig(output_path, bbox_inches="tight", dpi=150)
plt.close()

print(f"Saved SHAP visualization to: {output_path}")

LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(LOG_PATH, "a", encoding="utf-8") as log_file:
    log_file.write(f"=== Run {timestamp} ===\n")
    log_file.write(f"Checkpoint epoch: {checkpoint['epoch']}\n")
    log_file.write(
        f"Checkpoint val accuracy: {checkpoint['val_accuracy']:.4f}\n"
    )
    log_file.write(
        f"Test set misclassified: {test_misclassified_count}/"
        f"{test_total_count} ({test_misclassified_rate:.2%})\n"
    )
    log_file.write("\n".join(image_result_lines) + "\n")
    log_file.write(f"Result image: {output_path.resolve()}\n\n")

print(f"Logged run to: {LOG_PATH}")


Image 0: true=PNEUMONIA, predicted=PNEUMONIA
Image 1: true=PNEUMONIA, predicted=PNEUMONIA
Image 2: true=NORMAL, predicted=NORMAL
Image 3: true=NORMAL, predicted=NORMAL
Image 4: true=NORMAL, predicted=PNEUMONIA
Image 5: true=PNEUMONIA, predicted=NORMAL
Image 6: true=NORMAL, predicted=PNEUMONIA
Image 7: true=NORMAL, predicted=PNEUMONIA
Saved SHAP visualization to: /home/mega-pc/pneumo-xai/explanations/shap_epoch10_20261016_182353.png
Logged run to: /home/mega-pc/pneumo-xai/logs/shap_runs.log
